## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-20-21-54-44 +0000,nypost,Ice cream shop in elite California beach encla...,https://nypost.com/2026/11/20/business/popular...
1,2026-11-20-21-50-00 +0000,wsj,Nasdaq Turns Red For The Year,https://www.wsj.com/finance/stocks/nasdaq-turn...
2,2026-11-20-21-50-00 +0000,wsj,El Salvador Takes Its Gang-Fighting Strategy t...,https://www.wsj.com/world/americas/el-salvador...
3,2026-11-20-21-49-27 +0000,nypost,State Department official accused of lavish ‘v...,https://nypost.com/2026/11/20/us-news/state-de...
4,2026-11-20-21-47-21 +0000,nyt,Denmark Expels Russian Diplomat After Accusing...,https://www.nytimes.com/2026/11/20/world/europ...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2544/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
56,trump,46
142,anthropic,26
63,new,22
86,ai,19
45,tech,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
117,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,139
12,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,125
207,2026-11-20-14-28-00 +0000,wsj,The Trump administration auctioned off parts o...,https://www.wsj.com/politics/the-u-s-is-auctio...,106
103,2026-11-20-18-51-22 +0000,nyt,Trump Threatens to Bar U.S. Tech Firms From Bu...,https://www.nytimes.com/2026/11/20/us/politics...,101
64,2026-11-20-20-14-29 +0000,nypost,Trump backs Jensen Huang’s return as Nvidia CE...,https://nypost.com/2026/11/20/us-news/trump-ba...,101


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
117,139,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
78,76,2026-11-20-19-48-40 +0000,nypost,Anthropic’s new IPO millionaires awkward about...,https://nypost.com/2026/11/20/business/anthrop...
321,68,2026-11-20-04-12-00 +0000,wsj,The Senate passed a bipartisan bill designed t...,https://www.wsj.com/politics/policy/senate-pas...
86,66,2026-11-20-19-39-00 +0000,wsj,"Sen. Tom Cotton (R., Ark.), a China hawk, tell...",https://www.wsj.com/politics/national-security...
347,58,2026-11-20-01-11-52 +0000,latimes,California State Bar opens investigation into ...,https://www.latimes.com/california/story/2026-...
153,56,2026-11-20-16-41-52 +0000,bbc,Faisal Islam: This cost-cutting UK-EU deal doe...,https://www.bbc.co.uk/news/articles/cm1wr029ev...
47,52,2026-11-20-20-41-00 +0000,wsj,Take-Two’s ‘Grand Theft Auto VI’ Hits Nearly $...,https://www.wsj.com/business/media/take-twos-g...
225,46,2026-11-20-13-38-21 +0000,nypost,Jared Kushner says Russia appears ready to acc...,https://nypost.com/2026/11/20/us-news/jared-ku...
239,43,2026-11-20-12-00-00 +0000,wsj,Vice President JD Vance has whipsawed between ...,https://www.wsj.com/politics/trump-says-vance-...
331,36,2026-11-20-02-59-00 +0000,wsj,National Cyber Director Sean Cairncross is lea...,https://www.wsj.com/politics/policy/sean-cairn...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
